# ST554 Final Project: Siona Benjamin
For the final project we'll use spark to handle streaming data and fitting a machine learning model. The data used below describes power consumption from different time zones of Tetoauan city in relation to factors such as time of day, temperature, and humidity. 

To get started, we'll read in our data as a pandas dataframe, and then convert this to a spark dataframe.

In [25]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.regression import LinearRegression
from pyspark.sql.types import StructType

from pyspark.ml.feature import SQLTransformer, PCA, Binarizer, OneHotEncoder, VectorAssembler, StringIndexer

In [ ]:
#create spark session
spark = SparkSession.builder.appName("final_project").getOrCreate()

In [9]:
#import data as pandas dataframe
power_data = pd.read_csv('power_ml_data.csv')
#convert pandas dataframe to spark dataframe
power_df = spark.createDataFrame(power_data)

Using `.show()` we can see what our data columns look like while `.dtypes` lets us see what data type each column is stored as.

In [10]:
power_df.show(10)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
|      5.853|    76.9|     0.081|       

In [11]:
power_df.dtypes

[('Temperature', 'double'),
 ('Humidity', 'double'),
 ('Wind_Speed', 'double'),
 ('General_Diffuse_Flows', 'double'),
 ('Diffuse_Flows', 'double'),
 ('Power_Zone_1', 'double'),
 ('Power_Zone_2', 'double'),
 ('Power_Zone_3', 'double'),
 ('Month', 'bigint'),
 ('Hour', 'bigint')]

## Fitting the Model
The first part of this project will be training an elastic net model with out dataset to predict values for Power Zone 3. In an elastic net model, L1 (LASSO) and L2 (Ridge) penalties are combined to improve model predictions and stability. 

Now that we've loaded our dataset and have a good idea of what our data looks like, we can set up the transformations we want to apply to our data before training our model. The first transformation we'll apply is a SQL transformation to cast the Hour variable as a double instead of an integer. Within the same transformation, we will also set the Power_Zone_3 as label. After changing the Hour variable type, we'll apply a binarizer transformation to this variable to distinguish between night and day using 6.5 as the cutoff. Next , we'll use one-hot encoding to encode the Month variable. Additionally, we will run a PCA (Principle Component Analysis) fit on a few of the columns in our dataset. The PCA entails using a VectorAssembler transformation to place the desired variables together in a column followed by using the PCA transformation.

Lastly, we will use the VectorAssembler transformation to combine our desired predictor variables in a features column. 

In [16]:
#SQL transformer to cast Hour variable as DoubleType
sqlTrans = SQLTransformer(
    statement = """
                SELECT *,
                CAST(Hour AS DOUBLE) AS hour_double,
                Power_Zone_3 as label 
                FROM __THIS__
                """)

In [17]:
#Binarize transformer to convert continuous Hour values to binary values 
binarizer = Binarizer(threshold=6.5, inputCol="hour_double", outputCol="hour_binary")

In [14]:
#One-hot encoder to transform Month values to vector values
##StringIndexer transformation to conver Month values 
indexer = StringIndexer(inputCol="Month", outputCol="month_index")
##OneHotEncoder transformation
encoder = OneHotEncoder(inputCols=["Month"], outputCols=["month_vec"])

In [19]:
#PCA transformation 
##VectorAssembler to combine desired columns
pca_assembler = VectorAssembler(inputCols=["Temperature","Humidity","Wind_Speed","General_Diffuse_Flows","Diffuse_Flows"], outputCol="pca_features")
##PCA transformer 
pca = PCA(k=5,inputCol="pca_features", outputCol="pca_results")

Now that we have defined our transformations, we can define the other components of our model. First we'll create an object to define our linear regression model. Then we'll define our parameter grid to set test values of `regParam`, which controls the amount of regularization, and `elasticNetParam`, which defines the balance between L1 and L2 regularization. We will also set up a pipeline with the transformatin defined above and our linear regression model. 

In [22]:
#VectorAssembler to put predictors in features column 
assembler_features = VectorAssembler(inputCols=["hour_binary","Power_Zone_1","Power_Zone_2","month_vec","pca_results"], outputCol="features")

In [23]:
#define object for linear regression model 
lr = LinearRegression()
#define parameter grid 
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()
#define transformation pipeline 
pipeline = Pipeline(stages = [sqlTrans, binarizer, indexer, encoder, pca_assembler, pca, assembler_features, lr])

In [26]:
#set up cross validation 
crossval = CrossValidator(estimator = pipeline,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(metricName='rmse'),
                          numFolds=5)

In [28]:
cvModel = crossval.fit(power_df)

26/04/26 20:57:52 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/26 20:57:52 WARN Instrumentation: [3341ad6d] regParam is zero, which might cause numerical instability and overfitting.
26/04/26 20:57:53 WARN Instrumentation: [3341ad6d] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/26 20:57:53 ERROR LBFGS: Failure! Resetting history: breeze.optimize.FirstOrderException: Line search zoom failed
26/04/26 20:57:55 WARN Instrumentation: [52ae6066] regParam is zero, which might cause numerical instability and overfitting.
26/04/26 20:57:56 WARN Instrumentation: [52ae6066] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/26 20:57:57 WARN Instrumentation: [d7b655f6] regParam is zero, which might cause numerical instability and overfitting.
26/04/26 20:57:57 WARN In

In [33]:
lr_rmse = RegressionEvaluator().evaluate(cvModel.transform(power_df))

In [34]:
lr_rmse

2124.1528151933303

## Streaming Data
In the previous section we trained an elastic net model on our dataset to predict values of Power_Zone_3. With this model, we can now make predictions with new data that we read in from a stream. 

In [36]:
myschema = power_df.schema

In [ ]:
df = spark.readStream.schema(myschema).format("csv").option("header","true").load("streaming_files")

In [37]:
stream_data = pd.read_csv("power_streaming_data.csv")
stream_data.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,4.805,76.2,0.081,0.059,0.134,20421.26582,12908.20669,14590.84337,1,3
1,4.212,78.3,0.081,0.117,0.082,21393.41772,13575.68389,14862.65060,1,5
2,4.304,76.0,0.082,0.048,0.152,19983.79747,12342.85714,13492.04819,1,7
3,4.489,74.3,0.082,0.081,0.119,18167.08861,11551.36778,11600.96386,1,7
4,4.509,74.5,0.084,6.643,6.494,19837.97468,11945.28875,11178.79518,1,8


In [46]:
 %run produce_stream_data.py